# Tutorial: Estimating Social Contact Matrices with SocialMix

In [ ]:
# @title Setup (Google Colab only — skip if cntmosaic is already installed locally)
#
# Clones the cntmosaic repository, installs the package, and sets the working
# directory so that the relative data paths in this notebook resolve correctly.
import sys, os, subprocess

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab — cloning repository and installing cntmosaic...")
    subprocess.check_call(
        ["git", "clone", "--quiet", "https://github.com/ShozenD/cntmosaic.git"]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "./cntmosaic"]
    )
    os.chdir("cntmosaic/tutorials")
    print(f"Done. Working directory set to: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd

from cntmosaic.utils import AgeGroupSpecs
from cntmosaic.dataloader import ContactData, ParticipantData, PopulationData
from cntmosaic.vis import plot_mosaic_pixilated

from cntmosaic.models import SocialMix
from cntmosaic.analysis import ModelSummariserSocialMix

import altair as alt

In [ ]:
# ===== Participant data ======
# Load
df_part = pd.read_csv("data/POLYMOD_UK_participant_common.csv")

# Preprocess
df_part["part_age_grp"] = pd.cut(
    df_part["part_age"].astype("Int32"), bins=range(0, 81, 5), right=False
)
df_part["part_gender"] = pd.Categorical(df_part["part_gender"], categories=["M", "F"])

# ===== Contact data =====
# Load
df_cnt = pd.read_csv("data/POLYMOD_UK_contact_common.csv")

# Preprocess
df_cnt["cnt_age_grp"] = pd.cut(
    df_cnt["cnt_age_exact"].astype("Int32"), bins=range(0, 81, 5), right=False
)
df_cnt["cnt_gender"] = pd.Categorical(df_cnt["cnt_gender"], categories=["M", "F"])

# ===== Population data =====
# Load
df_pop = pd.read_csv("data/UK_population_2011.csv")
df_pop.head(10)

# Preprocess
df_pop["age"] = df_pop["age"].str.replace("+", "").astype(int)
df_pop = df_pop[df_pop["age"] <= 81].copy()
df_pop = df_pop.rename(columns={"Male": "M", "Female": "F"}).melt(
    id_vars="age", var_name="gender", value_name="P"
)
# df_pop["age_grp"] = pd.cut(df_pop["age"], bins=range(0, 81, 5), right=False)
# df_pop = df_pop.groupby(["age_grp", "gender"], observed=True)["P"].sum().reset_index()
df_pop["gender"] = pd.Categorical(df_pop["gender"], categories=["M", "F"])

In [ ]:
age_grp_specs = AgeGroupSpecs(0, 79, 5)
part_data = ParticipantData(df_part, id_col="part_id", age_grp_col="part_age_grp")
cnt_data = ContactData(df_cnt, id_col="part_id", age_grp_col="cnt_age_grp")
pop_data = PopulationData(df_pop, age_col="age", size_col="P")

sm = SocialMix(
    part_data=part_data,
    cnt_data=cnt_data,
    age_group_specs=age_grp_specs,
    pop_data=pop_data,
    apply_reciprocity=True,
    adaptive_merge=False,
)

In [ ]:
plot_mosaic_pixilated(
    sm.cint()["All->All"],
    xlabel="Age of respondent",
    ylabel="Age of contact",
    zlabel="Intensity",
)

In [ ]:
boot = sm.run_inference_bootstrap(n_boot=3000, random_state=1)

In [ ]:
summarizer = ModelSummariserSocialMix(sm)

In [ ]:
summ_cint = summarizer.summarise_cint()

plot_mosaic_pixilated(
    summ_cint["All->All"].upper - summ_cint["All->All"].lower,
    summ_cint["All->All"].age_group_specs,
    zlabel="Width of 95% Bootstrap CI",
    color_scheme="viridis",
)